![STREAMLINE](https://github.com/UrbsLab/STREAMLINE/blob/main/docs/source/pictures/STREAMLINE_Logo_Full.png?raw=true)

# STREAMLINE v3 Jupyter Notebook

STREAMLINE is an end-to-end automated machine learning workflow that helps users run, interpret, and apply a rigorous tabular-data analysis without hand-building every preprocessing, modeling, evaluation, and reporting step.

This local notebook can run one of the included UCI demos or a custom dataset from folders on your machine. It is meant to be run from the STREAMLINE repository root after installing project requirements.

Included demos:

- `demo_binary`: HCC Survival binary classification
- `demo_multiclass`: Student Dropout and Academic Success multiclass classification
- `demo_regression`: Auto MPG regression

Normal demo data folders contain deterministic 80% training splits; matching replication folders contain held-out 20% replication splits.


## Run Instructions

### Demo Run

1. Set `RUN_MODE` to `demo_binary`, `demo_multiclass`, or `demo_regression`.
2. Keep `OUTPUT_PATH = None` to write outputs to `<repo>/out`, or provide another folder.
3. Run cells from top to bottom.

The demo settings use 3 CV folds and compact model lists so the notebook can be used as a practical tutorial. You can optionally change non-dataset parameters below, such as phase toggles, CV folds, feature-importance methods, or modeling algorithms.

### Custom Dataset Run

1. Set `RUN_MODE` to `custom_classification` or `custom_regression`.
2. Update the `CUSTOM_*` paths and labels below.
3. Optional replication data can be supplied through `CUSTOM_REPLICATION_PATH` and `CUSTOM_DATASET_FOR_REP`.
4. Run cells from top to bottom.

Target data folders may contain one or more `.csv`, `.tsv`, or `.txt` datasets. All datasets in the same run should use the same outcome, instance, and match column names.

The top parameter cells are the normal editing surface. The phase cells are intentionally explicit so the notebook remains useful for teaching, debugging, and partial reruns.


In [ ]:
# [run] section: choose which built-in demo or custom workflow to run.
RUN_MODE = "demo_binary"
# Supported:
# - demo_binary
# - demo_multiclass
# - demo_regression
# - custom_classification
# - custom_regression
# Backward-compatible alias: demo_classification -> demo_multiclass

# [run] output_path / experiment_name
EXPERIMENT_NAME = "JupyterRun"
OUTPUT_PATH = None  # None -> defaults to <repo>/out


### Repository and Environment Parameters

Use these settings to control local environment behavior.

- `REPO_DIR_OVERRIDE`: explicit repo path if auto-detection fails.
- `INSTALL_REQUIREMENTS`: install from `requirements.txt` in the current Python environment.
- `RUN_CLUSTER`: `Serial` is simplest for notebooks; `Parallel` uses local joblib; `Local` uses local Dask.
- `P1_FORCE`: rebuilds the experiment folder when Phase 1 is rerun.

For small notebook demos, `Serial` is usually easiest to inspect. For larger runs, config/CLI execution is often a better fit.


In [ ]:
REPO_DIR_OVERRIDE = None
INSTALL_REQUIREMENTS = False

# [run] execution parameters
RUN_CLUSTER = "Serial"  # Serial, Parallel (joblib), Local (Dask), BashSLURM, BashLSF, or a named Dask cluster
QUEUE = "defq"
RESERVED_MEMORY = 4
N_SPLITS = 3
RANDOM_STATE = 42

# Kept as aliases because runner cells use PHASE_* names.
PHASE_QUEUE = QUEUE
PHASE_RESERVED_MEMORY_GB = RESERVED_MEMORY


### Custom Data Parameters

Used only for `custom_*` modes.

- target data folder
- optional replication data folder
- reference training dataset for replication mapping
- labels for outcome, instance id, and matching/grouping
- optional categorical/quantitative feature-list files or lists

If categorical/quantitative feature lists are left as `None`, STREAMLINE uses `P1_CATEGORICAL_CUTOFF` and binary-feature detection to infer feature types.


In [ ]:
CUSTOM_DATA_PATH = "/absolute/path/to/target_data_folder"
CUSTOM_REPLICATION_PATH = "/absolute/path/to/replication_data_folder"
CUSTOM_DATASET_FOR_REP = "/absolute/path/to/target_data_folder/train_dataset.csv"

CUSTOM_OUTCOME_LABEL = "Class"
CUSTOM_INSTANCE_LABEL = None
CUSTOM_MATCH_LABEL = None
CUSTOM_CATEGORICAL_FEATURES = None
CUSTOM_QUANTITATIVE_FEATURES = None

CUSTOM_CLASSIFICATION_TYPE = "Binary"   # Binary or Multiclass
CUSTOM_CLASSIFICATION_MODELS = "NB,LR,DT"
CUSTOM_REGRESSION_MODELS = "LR,RF"


----------------------
## Non-Essential Run Parameters

### Phase Toggles

Use toggles to rerun only selected phases during iteration. For a complete fresh run, leave everything `True`.

`ALLOW_P7_FOR_REGRESSION` controls whether the ensemble phase runs for regression workflows. Leave it `False` unless regression ensembles are explicitly supported for the run you are testing.

If you skip an upstream phase, make sure its expected output already exists from a previous run.


In [ ]:
# [phases] section
PHASE_ORDER = "p1,p2,p3,p4,p5,p6,p7,p8,p9,p10,p11"
RUN_PHASES = {
    "p1": True,
    "p2": True,
    "p3": True,
    "p4": True,
    "p5": True,
    "p6": True,
    "p7": True,
    "p8": True,
    "p9": True,
    "p10": True,
    "p11": True,
    "p11_replication": True,
}

# For regression, set True only if you explicitly want to run ensembles.
ALLOW_P7_FOR_REGRESSION = False


--------------
### Phase 1 - Data Exploration and Processing Parameters

Phase 1 performs initial EDA, cleaning, feature engineering, feature-type handling, and CV partitioning.

Typical outputs include data-count summaries, missingness summaries, class/outcome summaries, correlation plots, univariate analysis, and saved CV train/test files. Feature type settings matter downstream because categorical features may be one-hot encoded, passed to native categorical models, or passed to ReBATE methods as categorical indexes.

When `P1_CATEGORICAL_FEATURES` and `P1_QUANTITATIVE_FEATURES` are `None`, STREAMLINE infers feature type using `P1_CATEGORICAL_CUTOFF`: non-binary features with more unique values than the cutoff are treated as quantitative, while binary features are treated as categorical by default.


In [ ]:
# [p1] Data Exploration and Processing parameters
P1_EXCLUDE_EDA_OUTPUT = None
P1_PARTITION_METHOD = "Stratified"
P1_IGNORE_FEATURES = None
P1_CATEGORICAL_FEATURES = None
P1_QUANTITATIVE_FEATURES = None
P1_TOP_FEATURES = 20
P1_CATEGORICAL_CUTOFF = 10
P1_SIG_CUTOFF = 0.05
P1_FEATUREENG_MISSINGNESS = 0.5
P1_CLEANING_MISSINGNESS = 0.5
P1_CORRELATION_REMOVAL_THRESHOLD = 1.0
P1_SHOW_PLOTS = True
P1_ONE_HOT_ENCODING = True
P1_CV_PROVIDED = False
P1_CV_INPUT_ROOT = None
P1_ENABLE_PLOTS = True
P1_PLOT_MISSINGNESS = True
P1_PLOT_CLASS_COUNTS = True
P1_PLOT_CORRELATION = True
P1_CORRELATION_PLOT_MAX_FEATURES = 200
P1_PLOT_UNIVARIATE = True
P1_UNIVARIATE_TOP_K = 20
P1_PLOT_ANOMALIES = True
P1_FORCE = True


### Phase 2 - Impute, Scale, and Balance Parameters

Phase 2 transforms each CV split after Phase 1. Missing-value imputation and scaling are fit on training folds and applied to the corresponding test folds.

Optional SMOTE/SMOTENC oversampling is applied only to training folds and only for classification outcomes. With `P2_SMOTE_METHOD = "auto"`, STREAMLINE uses SMOTENC when categorical features are present and standard SMOTE otherwise.


In [ ]:
# [p2] Impute, Scale, and Balance parameters
P2_SCALE_DATA = True
P2_IMPUTE_DATA = True
P2_MULTI_IMPUTE = False
P2_OVERWRITE_CV = True
P2_IMPUTER_ID = None
P2_IMPUTER_PARAMS = {}
P2_SCALER_ID = None
P2_SCALER_PARAMS = {}
P2_SMOTE = False
P2_SMOTE_METHOD = "auto"
P2_SMOTE_SAMPLING_STRATEGY = "auto"
P2_SMOTE_K_NEIGHBORS = 5


### Phase 3 - Feature Learning Parameters

Phase 3 adds learned representations such as PCA features. Learned transformations are fit within each training fold and applied to the matching test fold, preserving CV separation.

`P3_KEEP_ORIGINAL_FEATURES=True` appends learned features to the original feature set. Set it to `False` when you want downstream phases to use only the learned representation.


In [ ]:
# [p3] Feature Learning parameters
P3_LEARNER_ID = "pca"
P3_LEARNER_PARAMS = {}
P3_FEATURE_NAMESPACE = "FL_PCA"
P3_KEEP_ORIGINAL_FEATURES = True
P3_OVERWRITE_CV = True


### Phase 4 - Feature Importance Parameters

Phase 4 estimates filter-based feature importance within each CV split. These scores are later used for feature selection and reporting.

The demo/config default uses Mutual Information and MultiSWRFDB. Set `P4_MODELS = None` to run every registered feature-importance method. `P4_INSTANCE_SUBSET` limits expensive ReBATE-style methods; leave it as `None` to use all training instances.


In [ ]:
# [p4] Feature Importance parameters
P4_MODELS = "mutualinformation,multiswrfdb"
P4_MODELS_PARAMS = {"mutualinformation": {"outcome_type": "auto"}, "multiswrfdb": {"n_jobs": 1}}
P4_TOP_K = None
P4_THRESHOLD = None
P4_KEEP_ORIGINAL_FEATURES = False
P4_OVERWRITE_CV = True
P4_INSTANCE_SUBSET = None


### Phase 5 - Feature Selection Parameters

Phase 5 performs collective feature selection before modeling. When `P5_FILTER_POOR_FEATURES = False`, all features continue to modeling.

When filtering is enabled, STREAMLINE removes features that have no evidence of importance across the active FI methods. If `P5_MAX_FEATURES_TO_KEEP` is an integer, STREAMLINE keeps a capped set of top-ranked features from the available FI outputs. `P5_ALGORITHMS = "auto"` is recommended because it discovers whichever FI methods were actually run.


In [ ]:
# [p5] Feature Selection parameters
P5_ALGORITHMS = "auto"
P5_N_SPLITS = N_SPLITS
P5_MAX_FEATURES_TO_KEEP = 2000
P5_FILTER_POOR_FEATURES = True
P5_OVERWRITE_CV = False
P5_SELECTOR_ID = "default"
P5_SELECTOR_PARAMS = {}
P5_EXPORT_SCORES = True
P5_TOP_FEATURES = 20
P5_SHOW_PLOTS = True
P5_STRICT_DISCOVERY = False


### Phase 6 - Modeling Parameters

Phase 6 trains and evaluates the requested algorithms across CV folds.

- `P6_N_TRIALS` and `P6_TIMEOUT` control Optuna hyperparameter search. The notebook demo default is `50` trials and `300` seconds so Colab runs stay practical; use `200` and `900` for fuller config-style runs.
- `P6_TRAINING_SUBSAMPLE` can limit expensive models on large datasets.
- `P6_UNIFORM_FI=True` forces permutation feature importance for all models and can be slow.
- `P6_CALIBRATE=True` applies probability calibration for classification models.

The demo model lists are intentionally small (`NB,LR,DT` for classification and `LR,RF` for regression). Add more models when runtime is acceptable. The notebook defaults are tuned for demo/tutorial runtime; increase `P6_N_TRIALS` and `P6_TIMEOUT` for fuller local research runs. If Phase 1 one-hot encoding is disabled, requested models should be native-categorical compatible.


In [ ]:
# [p6] Modeling parameters
# None values below use the selected RUN_MODE defaults from CFG.
P6_OUTCOME_TYPE = None
P6_MODEL_TYPE = None
P6_MODELS = None
P6_MODEL_PARAMS_JSON = None
P6_CALIBRATE = False
P6_CALIBRATE_METHOD = "sigmoid"
P6_CALIBRATE_CV = 5
P6_SCORING_METRIC = None
P6_METRIC_DIRECTION = "maximize"
P6_N_TRIALS = 50
P6_TIMEOUT = 300
P6_TRAINING_SUBSAMPLE = 0
P6_UNIFORM_FI = False
P6_SAVE_PLOT = False
P6_BYPASS_ONE_HOT_FOR_NATIVE_MODELS = False
P6_NATIVE_CATEGORICAL_MODELS = "CGB,ExSTraCS"


### Phase 7 - Ensemble Parameters

Phase 7 builds ensemble classifiers from Phase 6 base-model predictions. Hard voting, soft voting, and stacking are available for classification workflows.

Regression ensemble support is intentionally guarded in this notebook by `ALLOW_P7_FOR_REGRESSION`; leave that off unless regression ensemble support is explicitly implemented for the run you are testing.


In [ ]:
# [p7] Ensemble parameters
P7_ENABLED = True
P7_ENSEMBLES = "hard_voting,soft_voting,stack_lr"
P7_BASE_MODELS = None  # None uses the selected RUN_MODE Phase 6 model list.
P7_META_TRAIN_SOURCE = "train"
P7_CALIBRATE = 0
P7_CALIBRATE_METHOD = "sigmoid"
P7_CALIBRATE_CV = 5


### Phase 8 - Summary Statistics Parameters

Phase 8 summarizes testing-fold model performance and creates post-analysis figures. Outputs may include ROC/PRC plots for classification, regression summaries, metric boxplots, model feature-importance plots, and composite feature-importance summaries.

`P8_MULTICLASS_AVERAGE` controls multiclass ROC/PR aggregation where applicable.


In [ ]:
# [p8] Summary Statistics parameters
# None metric values use the selected RUN_MODE defaults from CFG.
P8_SCORING_METRIC = None
P8_METRIC_WEIGHT = None
P8_TOP_FEATURES = 40
P8_SIG_CUTOFF = 0.05
P8_SCALE_DATA = True
P8_EXCLUDE_PLOTS = None
P8_SHOW_PLOTS = True
P8_INCLUDE_ENSEMBLES = None  # None follows whether Phase 7 actually ran.
P8_MULTICLASS_AVERAGE = "micro"


### Phase 9 - Dataset Comparison Parameters

Phase 9 is useful only when an experiment includes more than one target dataset. It compares model performance distributions across datasets and writes comparison outputs under `DatasetComparisons/`.


In [ ]:
# [p9] Dataset Comparison parameters
P9_SIG_CUTOFF = 0.05
P9_SHOW_PLOTS = True


### Phase 10 - Replication Parameters

Phase 10 evaluates previously trained models on external or held-out replication datasets. This gives a uniform evaluation on the same new data and can be more useful for selecting a final model than comparing only CV test folds.

The included UCI demos use deterministic held-out 20% replication splits rather than independent external datasets.


In [ ]:
# [p10] Replication parameters
# None path/label values use the selected RUN_MODE defaults from CFG/metadata.
P10_REP_DATA_PATH = None
P10_DATASET_FOR_REP = None
P10_MATCH_LABEL = None
P10_OUTCOME_LABEL = None
P10_INSTANCE_LABEL = None
P10_EXCLUDE_PLOTS = None
P10_SHOW_PLOTS = True


### Phase 11 - Reporting Parameters

Phase 11 generates PDF report artifacts for the standard testing-data report and, when available, the replication report.

`P11_REUSE_EXISTING_FIGURES=True` speeds up report regeneration when plots have already been made. Disable it when you want plots regenerated from scratch.


In [ ]:
# [p11] Reporting parameters
P11_REPORT_MODES = "standard,replication"
P11_REPORTING_DIR = None
P11_REPORT_MODE_STANDARD = "standard"
P11_REPORT_MODE_REPLICATION = "replication"
P11_OUTCOME_LABEL = None
P11_OUTCOME_TYPE = None
P11_INSTANCE_LABEL = None
P11_MAKE_PDF = True
P11_ENABLE_PLOTS = True
P11_REUSE_EXISTING_FIGURES = True


-------------
## Notebook Housekeeping

This setup section resolves paths, optionally installs dependencies, imports phase runners, and prepares logging.

Most users should not edit below this point unless they are debugging the notebook itself. If dependency versions look wrong, restart the kernel after installation and rerun from the top.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import logging

cwd = Path.cwd().resolve()

if REPO_DIR_OVERRIDE:
    REPO_DIR = Path(REPO_DIR_OVERRIDE).resolve()
else:
    # Auto-detect repo root by finding a directory that contains "streamline"
    if (cwd / "streamline").is_dir():
        REPO_DIR = cwd
    elif (cwd.parent / "streamline").is_dir():
        REPO_DIR = cwd.parent
    else:
        raise FileNotFoundError(
            "Could not auto-detect repo root. Set REPO_DIR_OVERRIDE to the STREAMLINE repo path."
        )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if INSTALL_REQUIREMENTS:
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)


import importlib.metadata as importlib_metadata

print("Dependency versions:")
for package_name in ["numpy", "pandas", "skrebate", "kaleido"]:
    try:
        version = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        version = "not installed"
    print(f"{package_name}: {version}")

# Show full phase logs in notebook output (INFO/WARNING/ERROR).
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
    force=True,
)
logging.getLogger().setLevel(logging.INFO)
print("INFO: Notebook logging configured at INFO level (phases 1-10).")


from streamline.p1_data_process.p1_runner import P1Runner
from streamline.p2_impute_scale.p2_runner import P2Runner
from streamline.p3_feature_learning.p3_runner import P3Runner
from streamline.p4_feature_importance.p4_runner import P4Runner
from streamline.p5_feature_selection.p5_runner import P5Runner
from streamline.p6_modeling.p6_runner import P6Runner
from streamline.p7_ensembles.p7_runner import P7Runner
from streamline.p8_summary_statistics.p8_runner import P8Runner
from streamline.p9_compare_datasets.p9_runner import P9Runner
from streamline.p10_replication.p10_runner import P10Runner
from streamline.p11_reporting.p11_runner import P11Runner

if OUTPUT_PATH is None:
    OUTPUT_PATH = str((REPO_DIR / "out").resolve())

Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)
print(f"REPO_DIR: {REPO_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

In [ ]:
%matplotlib inline

## Build Mode-Specific Configuration

This cell converts `RUN_MODE` into a concrete configuration dictionary (`CFG`) including paths, labels, task type, feature-list defaults, model defaults, and metric defaults.

For demo modes, these values point to the UCI demo folders shipped with STREAMLINE. For custom modes, they come from the manual `CUSTOM_*` cells.


In [ ]:
from pprint import pprint


def uci_feature_path(repo_dir: Path, filename: str) -> str:
    return str(repo_dir / "data" / "UCIFeatureTypes" / filename)


def mode_defaults(run_mode: str, repo_dir: Path):
    mode = run_mode.strip().lower()
    if mode == "demo_classification":
        mode = "demo_multiclass"

    if mode == "demo_binary":
        return {
            "task_family": "classification",
            "data_path": str(repo_dir / "data" / "UCIBinaryClassification"),
            "rep_data_path": str(repo_dir / "data" / "UCIRepBinaryClassification"),
            "dataset_for_rep": str(repo_dir / "data" / "UCIBinaryClassification" / "hcc_survival.csv"),
            "outcome_label": "Class",
            "instance_label": "InstanceID",
            "match_label": None,
            "outcome_type": "Binary",
            "categorical_features": uci_feature_path(repo_dir, "hcc_survival_categorical_features.csv"),
            "quantitative_features": uci_feature_path(repo_dir, "hcc_survival_quantitative_features.csv"),
            "p6_models": "NB,LR,DT",
            "p6_scoring_metric": "balanced_accuracy",
            "p6_metric_direction": "maximize",
            "p8_scoring_metric": "balanced_accuracy",
            "p8_metric_weight": "balanced_accuracy",
        }

    if mode == "demo_multiclass":
        return {
            "task_family": "classification",
            "data_path": str(repo_dir / "data" / "UCIMulticlassClassification"),
            "rep_data_path": str(repo_dir / "data" / "UCIRepMulticlassClassification"),
            "dataset_for_rep": str(repo_dir / "data" / "UCIMulticlassClassification" / "student_dropout_academic_success.csv"),
            "outcome_label": "Class",
            "instance_label": "InstanceID",
            "match_label": None,
            "outcome_type": "Multiclass",
            "categorical_features": uci_feature_path(repo_dir, "student_dropout_categorical_features.csv"),
            "quantitative_features": uci_feature_path(repo_dir, "student_dropout_quantitative_features.csv"),
            "p6_models": "NB,LR,DT",
            "p6_scoring_metric": "balanced_accuracy",
            "p6_metric_direction": "maximize",
            "p8_scoring_metric": "balanced_accuracy",
            "p8_metric_weight": "balanced_accuracy",
        }

    if mode == "demo_regression":
        return {
            "task_family": "regression",
            "data_path": str(repo_dir / "data" / "UCIRegression"),
            "rep_data_path": str(repo_dir / "data" / "UCIRepRegression"),
            "dataset_for_rep": str(repo_dir / "data" / "UCIRegression" / "auto_mpg.csv"),
            "outcome_label": "MPG",
            "instance_label": "InstanceID",
            "match_label": None,
            "outcome_type": "Continuous",
            "categorical_features": uci_feature_path(repo_dir, "auto_mpg_categorical_features.csv"),
            "quantitative_features": uci_feature_path(repo_dir, "auto_mpg_quantitative_features.csv"),
            "p6_models": "LR,RF",
            "p6_scoring_metric": "neg_mean_squared_error",
            "p6_metric_direction": "maximize",
            "p8_scoring_metric": "mean_squared_error",
            "p8_metric_weight": "mean_squared_error",
        }

    if mode == "custom_classification":
        ctype = CUSTOM_CLASSIFICATION_TYPE.strip()
        if ctype not in {"Binary", "Multiclass"}:
            raise ValueError("CUSTOM_CLASSIFICATION_TYPE must be Binary or Multiclass")
        return {
            "task_family": "classification",
            "data_path": CUSTOM_DATA_PATH,
            "rep_data_path": CUSTOM_REPLICATION_PATH,
            "dataset_for_rep": CUSTOM_DATASET_FOR_REP,
            "outcome_label": CUSTOM_OUTCOME_LABEL,
            "instance_label": CUSTOM_INSTANCE_LABEL,
            "match_label": CUSTOM_MATCH_LABEL,
            "outcome_type": ctype,
            "categorical_features": CUSTOM_CATEGORICAL_FEATURES,
            "quantitative_features": CUSTOM_QUANTITATIVE_FEATURES,
            "p6_models": CUSTOM_CLASSIFICATION_MODELS,
            "p6_scoring_metric": "balanced_accuracy",
            "p6_metric_direction": "maximize",
            "p8_scoring_metric": "balanced_accuracy",
            "p8_metric_weight": "balanced_accuracy",
        }

    if mode == "custom_regression":
        return {
            "task_family": "regression",
            "data_path": CUSTOM_DATA_PATH,
            "rep_data_path": CUSTOM_REPLICATION_PATH,
            "dataset_for_rep": CUSTOM_DATASET_FOR_REP,
            "outcome_label": CUSTOM_OUTCOME_LABEL,
            "instance_label": CUSTOM_INSTANCE_LABEL,
            "match_label": CUSTOM_MATCH_LABEL,
            "outcome_type": "Continuous",
            "categorical_features": CUSTOM_CATEGORICAL_FEATURES,
            "quantitative_features": CUSTOM_QUANTITATIVE_FEATURES,
            "p6_models": CUSTOM_REGRESSION_MODELS,
            "p6_scoring_metric": "neg_mean_squared_error",
            "p6_metric_direction": "maximize",
            "p8_scoring_metric": "mean_squared_error",
            "p8_metric_weight": "mean_squared_error",
        }

    raise ValueError(f"Unsupported RUN_MODE: {run_mode}")


CFG = mode_defaults(RUN_MODE, REPO_DIR)
CFG["output_path"] = OUTPUT_PATH
CFG["experiment_name"] = EXPERIMENT_NAME

# Explicit Phase 1 feature-list parameters override mode defaults when provided.
if P1_CATEGORICAL_FEATURES is not None:
    CFG["categorical_features"] = P1_CATEGORICAL_FEATURES
if P1_QUANTITATIVE_FEATURES is not None:
    CFG["quantitative_features"] = P1_QUANTITATIVE_FEATURES

if not Path(CFG["data_path"]).exists():
    raise FileNotFoundError(f"Target data path not found: {CFG['data_path']}")

EXP_ROOT = Path(CFG["output_path"]) / CFG["experiment_name"]

print("Resolved config:")
pprint(CFG)


-------------
# STREAMLINE RUN CODE
The cells below run STREAMLINE phase-by-phase.

For full runs, execute in order.
For debugging, toggle phases and rerun only downstream sections.

## Phase 1: Data Exploration and Processing
After this cell runs, each target dataset should have:

- initial EDA data counts and missingness summaries
- notifications for removed features/instances and added engineered features
- processed-data EDA summaries
- class/outcome balance plots when applicable
- feature correlation and univariate analysis outputs
- CV train/test partitions and feature metadata for downstream phases


In [ ]:
if RUN_PHASES["p1"]:
    print("INFO: Starting Phase 1 - Data Processing")
    P1Runner(
        data_path=CFG["data_path"],
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        exclude_eda_output=P1_EXCLUDE_EDA_OUTPUT,
        outcome_label=CFG["outcome_label"],
        outcome_type=CFG["outcome_type"],
        instance_label=CFG["instance_label"],
        match_label=CFG["match_label"],
        n_splits=N_SPLITS,
        partition_method=P1_PARTITION_METHOD,
        ignore_features=P1_IGNORE_FEATURES,
        categorical_features=CFG.get("categorical_features"),
        quantitative_features=CFG.get("quantitative_features"),
        top_features=P1_TOP_FEATURES,
        categorical_cutoff=P1_CATEGORICAL_CUTOFF,
        sig_cutoff=P1_SIG_CUTOFF,
        featureeng_missingness=P1_FEATUREENG_MISSINGNESS,
        cleaning_missingness=P1_CLEANING_MISSINGNESS,
        correlation_removal_threshold=P1_CORRELATION_REMOVAL_THRESHOLD,
        random_state=RANDOM_STATE,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
        show_plots=P1_SHOW_PLOTS,
        one_hot_encoding=P1_ONE_HOT_ENCODING,
        cv_provided=P1_CV_PROVIDED,
        cv_input_root=P1_CV_INPUT_ROOT,
        enable_plots=P1_ENABLE_PLOTS,
        plot_missingness=P1_PLOT_MISSINGNESS,
        plot_class_counts=P1_PLOT_CLASS_COUNTS,
        plot_correlation=P1_PLOT_CORRELATION,
        correlation_plot_max_features=P1_CORRELATION_PLOT_MAX_FEATURES,
        plot_univariate=P1_PLOT_UNIVARIATE,
        univariate_top_k=P1_UNIVARIATE_TOP_K,
        plot_anomalies=P1_PLOT_ANOMALIES,
        force=P1_FORCE,
    ).run()
    print("INFO: Completed Phase 1 - Data Processing")
else:
    print("INFO: Phase 1 skipped")


## Phase 2: Impute, Scale, and Balance
After this cell runs, CV datasets are transformed according to Phase 2 settings:

- missing values are imputed
- quantitative features are optionally scaled
- optional SMOTE/SMOTENC balancing is applied to training folds only

The test fold is transformed using objects learned from the training fold to preserve valid CV separation.


In [ ]:
if RUN_PHASES["p2"]:
    print("INFO: Starting Phase 2 - Scaling and Imputation")
    P2Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        scale_data=P2_SCALE_DATA,
        impute_data=P2_IMPUTE_DATA,
        multi_impute=P2_MULTI_IMPUTE,
        overwrite_cv=P2_OVERWRITE_CV,
        outcome_label=CFG["outcome_label"],
        instance_label=CFG["instance_label"],
        random_state=RANDOM_STATE,
        imputer_id=P2_IMPUTER_ID,
        imputer_params=P2_IMPUTER_PARAMS,
        scaler_id=P2_SCALER_ID,
        scaler_params=P2_SCALER_PARAMS,
        smote=P2_SMOTE,
        smote_method=P2_SMOTE_METHOD,
        smote_sampling_strategy=P2_SMOTE_SAMPLING_STRATEGY,
        smote_k_neighbors=P2_SMOTE_K_NEIGHBORS,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
    ).run()
    print("INFO: Completed Phase 2 - Scaling and Imputation")
else:
    print("INFO: Phase 2 skipped")


## Phase 3: Feature Learning
After this cell runs, learned representations such as PCA features are created per CV split. These learned features can replace or augment the original features depending on `P3_KEEP_ORIGINAL_FEATURES`.


In [ ]:
if RUN_PHASES["p3"]:
    print("INFO: Starting Phase 3 - Feature Learning")
    P3Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        learner_id=P3_LEARNER_ID,
        learner_params=P3_LEARNER_PARAMS,
        feature_namespace=P3_FEATURE_NAMESPACE,
        keep_original_features=P3_KEEP_ORIGINAL_FEATURES,
        overwrite_cv=P3_OVERWRITE_CV,
        outcome_label=CFG["outcome_label"],
        instance_label=CFG["instance_label"],
        random_state=RANDOM_STATE,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
    ).run()
    print("INFO: Completed Phase 3 - Feature Learning")
else:
    print("INFO: Phase 3 skipped")


## Phase 4: Feature Importance
After this cell runs, feature-importance scores are saved for each active method and CV split. These outputs feed Phase 5 feature selection, summary statistics, and report tables/figures.


In [ ]:
if RUN_PHASES["p4"]:
    p4_model_params = P4_MODELS_PARAMS
    if isinstance(p4_model_params, dict):
        p4_model_params = {k: dict(v) for k, v in p4_model_params.items()}
        if p4_model_params.get("mutualinformation", {}).get("outcome_type") == "auto":
            p4_model_params["mutualinformation"]["outcome_type"] = CFG["outcome_type"]
    print("INFO: Starting Phase 4 - Feature Importance")
    P4Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        models=P4_MODELS,
        models_params=p4_model_params,
        top_k=P4_TOP_K,
        threshold=P4_THRESHOLD,
        keep_original_features=P4_KEEP_ORIGINAL_FEATURES,
        overwrite_cv=P4_OVERWRITE_CV,
        outcome_label=CFG["outcome_label"],
        outcome_type=CFG["outcome_type"],
        instance_label=CFG["instance_label"],
        random_state=RANDOM_STATE,
        instance_subset=P4_INSTANCE_SUBSET,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
    ).run()
    print("INFO: Completed Phase 4 - Feature Importance")
else:
    print("INFO: Phase 4 skipped")


## Phase 5: Feature Selection
After this cell runs, selected feature subsets and informative feature summaries are generated. If filtering is disabled, all features continue to modeling; otherwise STREAMLINE uses the FI outputs to remove weakly supported features.


In [ ]:
if RUN_PHASES["p5"]:
    print("INFO: Starting Phase 5 - Feature Selection")
    P5Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        algorithms=P5_ALGORITHMS,
        n_splits=P5_N_SPLITS,
        outcome_label=CFG["outcome_label"],
        instance_label=CFG["instance_label"],
        max_features_to_keep=P5_MAX_FEATURES_TO_KEEP,
        filter_poor_features=P5_FILTER_POOR_FEATURES,
        overwrite_cv=P5_OVERWRITE_CV,
        selector_id=P5_SELECTOR_ID,
        selector_params=P5_SELECTOR_PARAMS,
        export_scores=P5_EXPORT_SCORES,
        top_features=P5_TOP_FEATURES,
        show_plots=P5_SHOW_PLOTS,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
        strict_discovery=P5_STRICT_DISCOVERY,
    ).run()
    print("INFO: Completed Phase 5 - Feature Selection")
else:
    print("INFO: Phase 5 skipped")


## Phase 6: Modeling
After this cell runs, STREAMLINE trains and evaluates each requested model across CV splits. Outputs include trained model pickles, metrics, ROC/PRC or regression artifacts, Optuna trial summaries, and model feature-importance estimates.


In [ ]:
if RUN_PHASES["p6"]:
    print("INFO: Starting Phase 6 - Modeling")
    P6Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        outcome_label=CFG["outcome_label"],
        outcome_type=P6_OUTCOME_TYPE or CFG["outcome_type"],
        model_type=P6_MODEL_TYPE,
        instance_label=CFG["instance_label"],
        n_splits=N_SPLITS,
        models=P6_MODELS or CFG["p6_models"],
        model_params_json=P6_MODEL_PARAMS_JSON,
        calibrate=P6_CALIBRATE,
        calibrate_method=P6_CALIBRATE_METHOD,
        calibrate_cv=P6_CALIBRATE_CV,
        scoring_metric=P6_SCORING_METRIC or CFG["p6_scoring_metric"],
        metric_direction=P6_METRIC_DIRECTION or CFG["p6_metric_direction"],
        n_trials=P6_N_TRIALS,
        timeout=P6_TIMEOUT,
        training_subsample=P6_TRAINING_SUBSAMPLE,
        uniform_fi=P6_UNIFORM_FI,
        save_plot=P6_SAVE_PLOT,
        random_state=RANDOM_STATE,
        bypass_one_hot_for_native_models=P6_BYPASS_ONE_HOT_FOR_NATIVE_MODELS,
        native_categorical_models=P6_NATIVE_CATEGORICAL_MODELS,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
    ).run()
    print("INFO: Completed Phase 6 - Modeling")
else:
    print("INFO: Phase 6 skipped")


## Phase 7: Ensembles
Optional phase. After this cell runs, ensemble metrics and curves are generated from Phase 6 base models when enabled. The notebook skips this phase for regression unless `ALLOW_P7_FOR_REGRESSION=True`.


In [ ]:
print("INFO: Evaluating Phase 7 - Ensembles")
run_p7 = RUN_PHASES["p7"] and P7_ENABLED
if CFG["task_family"] == "regression" and not ALLOW_P7_FOR_REGRESSION:
    run_p7 = False
    print("INFO: Phase 7 skipped for regression (ALLOW_P7_FOR_REGRESSION=False)")

if run_p7:
    print("INFO: Starting Phase 7 - Ensembles")
    P7Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        n_splits=N_SPLITS,
        outcome_label=CFG["outcome_label"],
        instance_label=CFG["instance_label"],
        ensembles=P7_ENSEMBLES,
        base_models=(P7_BASE_MODELS or CFG["p6_models"]),
        meta_train_source=P7_META_TRAIN_SOURCE,
        calibrate=P7_CALIBRATE,
        calibrate_method=P7_CALIBRATE_METHOD,
        calibrate_cv=P7_CALIBRATE_CV,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
        random_state=RANDOM_STATE,
    ).run()
    print("INFO: Completed Phase 7 - Ensembles")
elif RUN_PHASES["p7"]:
    print("INFO: Phase 7 skipped")


## Phase 8: Summary Statistics
After this cell runs, STREAMLINE creates per-dataset performance summaries and post-analysis figures, including model comparison plots and feature-importance summaries used by the final report.


In [ ]:
if RUN_PHASES["p8"]:
    print("INFO: Starting Phase 8 - Summary Statistics")
    P8Runner(
        output_path=CFG["output_path"],
        experiment_name=CFG["experiment_name"],
        outcome_label=CFG["outcome_label"],
        outcome_type=CFG["outcome_type"],
        instance_label=CFG["instance_label"],
        n_splits=N_SPLITS,
        scoring_metric=P8_SCORING_METRIC or CFG["p8_scoring_metric"],
        metric_weight=P8_METRIC_WEIGHT or CFG["p8_metric_weight"],
        top_features=P8_TOP_FEATURES,
        sig_cutoff=P8_SIG_CUTOFF,
        scale_data=P8_SCALE_DATA,
        exclude_plots=P8_EXCLUDE_PLOTS,
        show_plots=P8_SHOW_PLOTS,
        include_ensembles=run_p7 if P8_INCLUDE_ENSEMBLES is None else P8_INCLUDE_ENSEMBLES,
        multiclass_average=P8_MULTICLASS_AVERAGE,
        run_cluster=RUN_CLUSTER,
        queue=PHASE_QUEUE,
        reserved_memory=PHASE_RESERVED_MEMORY_GB,
    ).run()
    print("INFO: Completed Phase 8 - Summary Statistics")
else:
    print("INFO: Phase 8 skipped")


## Phase 9: Dataset Comparison
Optional phase, used only when two or more target datasets were analyzed. It compares performance distributions across datasets and saves outputs under `DatasetComparisons/`.


In [ ]:
def list_dataset_dirs(exp_root: Path):
    ignore = {"jobs", "logs", "jobsCompleted", "dask_logs", "DatasetComparisons", "reporting", "reporting_replication"}
    if not exp_root.exists():
        return []
    return [
        p for p in sorted(exp_root.iterdir())
        if p.is_dir() and p.name not in ignore and (p / "CVDatasets").is_dir()
    ]

if RUN_PHASES["p9"]:
    print("INFO: Starting Phase 9 - Dataset Comparison")
    ds_count = len(list_dataset_dirs(EXP_ROOT))
    if ds_count >= 2:
        P9Runner(
            output_path=CFG["output_path"],
            experiment_name=CFG["experiment_name"],
            outcome_label=CFG["outcome_label"],
            outcome_type=CFG["outcome_type"],
            instance_label=CFG["instance_label"],
            sig_cutoff=P9_SIG_CUTOFF,
            show_plots=P9_SHOW_PLOTS,
            run_cluster=RUN_CLUSTER,
            queue=PHASE_QUEUE,
            reserved_memory=PHASE_RESERVED_MEMORY_GB,
        ).run()
        print("INFO: Completed Phase 9 - Dataset Comparison")
    else:
        print(f"INFO: Phase 9 skipped - requires >=2 datasets, found {ds_count}")
else:
    print("INFO: Phase 9 skipped")


## Phase 10: Replication
Optional phase, used when replication data is available. Trained models are re-evaluated on the same replication dataset(s), giving a uniform external or held-out performance check.


In [ ]:
if RUN_PHASES["p10"]:
    print("INFO: Starting Phase 10 - Replication")
    rep_path = Path(P10_REP_DATA_PATH or CFG["rep_data_path"])
    data_for_rep = Path(P10_DATASET_FOR_REP or CFG["dataset_for_rep"])
    p10_match_label = P10_MATCH_LABEL if P10_MATCH_LABEL is not None else CFG["match_label"]
    if rep_path.exists() and data_for_rep.exists():
        P10Runner(
            rep_data_path=str(rep_path),
            dataset_for_rep=str(data_for_rep),
            output_path=CFG["output_path"],
            experiment_name=CFG["experiment_name"],
            outcome_label=P10_OUTCOME_LABEL,
            instance_label=P10_INSTANCE_LABEL,
            match_label=p10_match_label,
            exclude_plots=P10_EXCLUDE_PLOTS,
            run_cluster=RUN_CLUSTER,
            queue=PHASE_QUEUE,
            reserved_memory=PHASE_RESERVED_MEMORY_GB,
            show_plots=P10_SHOW_PLOTS,
        ).run()
        print("INFO: Completed Phase 10 - Replication")
    else:
        print("INFO: Phase 10 skipped - replication path or dataset_for_rep not found")
else:
    print("INFO: Phase 10 skipped")


## Phase 11: Reporting
After this cell runs, report artifacts are generated:

- standard testing-data report under `reporting/`
- replication report under `reporting_replication/` when Phase 10 was run

The reports summarize actual run settings, data-processing changes, feature engineering/selection, model performance, and replication results where available.


In [ ]:
# Keep phase logs verbose globally, but reduce report-phase noise.
_report_logger = logging.getLogger()
_prev_log_level = _report_logger.level
if RUN_PHASES["p11"] or RUN_PHASES["p11_replication"]:
    _report_logger.setLevel(logging.WARNING)

p11_report_modes = {m.strip().lower() for m in str(P11_REPORT_MODES).split(",") if m.strip()}

try:
    if RUN_PHASES["p11"] and "standard" in p11_report_modes:
        print("INFO: Starting Phase 11 - Reporting (standard)")
        P11Runner(
            output_path=CFG["output_path"],
            experiment_name=CFG["experiment_name"],
            experiment_path=str(EXP_ROOT),
            reporting_dir=P11_REPORTING_DIR,
            report_mode=P11_REPORT_MODE_STANDARD,
            outcome_label=P11_OUTCOME_LABEL or CFG["outcome_label"],
            outcome_type=P11_OUTCOME_TYPE or CFG["outcome_type"],
            instance_label=P11_INSTANCE_LABEL if P11_INSTANCE_LABEL is not None else CFG["instance_label"],
            make_pdf=P11_MAKE_PDF,
            enable_plots=P11_ENABLE_PLOTS,
            reuse_existing_figures=P11_REUSE_EXISTING_FIGURES,
            run_cluster=RUN_CLUSTER,
            queue=PHASE_QUEUE,
            reserved_memory=PHASE_RESERVED_MEMORY_GB,
        ).run()
        print("INFO: Completed Phase 11 - Reporting (standard)")
    else:
        print("INFO: Standard report skipped")

    if RUN_PHASES["p11_replication"] and "replication" in p11_report_modes:
        print("INFO: Starting Phase 11 - Reporting (replication)")
        P11Runner(
            output_path=CFG["output_path"],
            experiment_name=CFG["experiment_name"],
            experiment_path=str(EXP_ROOT),
            reporting_dir=P11_REPORTING_DIR,
            report_mode=P11_REPORT_MODE_REPLICATION,
            outcome_label=P11_OUTCOME_LABEL or CFG["outcome_label"],
            outcome_type=P11_OUTCOME_TYPE or CFG["outcome_type"],
            instance_label=P11_INSTANCE_LABEL if P11_INSTANCE_LABEL is not None else CFG["instance_label"],
            make_pdf=P11_MAKE_PDF,
            enable_plots=P11_ENABLE_PLOTS,
            reuse_existing_figures=P11_REUSE_EXISTING_FIGURES,
            run_cluster=RUN_CLUSTER,
            queue=PHASE_QUEUE,
            reserved_memory=PHASE_RESERVED_MEMORY_GB,
        ).run()
        print("INFO: Completed Phase 11 - Reporting (replication)")
    else:
        print("INFO: Replication report skipped")
finally:
    _report_logger.setLevel(_prev_log_level)

print("INFO: Pipeline run complete")


## Output Summary
This final check prints the experiment folder and expected report paths. If a report path says `exists=False`, verify that Phase 11 ran and that earlier phases completed successfully.


In [ ]:
exp_root = Path(CFG["output_path"]) / CFG["experiment_name"]
std_pdf = exp_root / "reporting" / f"{CFG['experiment_name']}_STREAMLINE_Report.pdf"
rep_pdf = exp_root / "reporting_replication" / f"{CFG['experiment_name']}_STREAMLINE_Replication_Report.pdf"

print(f"Experiment root: {exp_root}")
print(f"Standard report:   {std_pdf} (exists={std_pdf.exists()})")
print(f"Replication report:{rep_pdf} (exists={rep_pdf.exists()})")

## Optional: Zip Experiment Folder

Run this cell when you want a single archive of the experiment folder for sharing, moving results, or attaching outputs to an issue/tutorial record.


In [ ]:
import shutil
from pathlib import Path

exp_root = Path(CFG["output_path"]) / CFG["experiment_name"]
zip_base = Path(CFG["output_path"]) / f"{CFG['experiment_name']}_results"

if exp_root.exists():
    archive_path = shutil.make_archive(str(zip_base), "zip", exp_root)
    print(f"Created archive: {archive_path}")
else:
    print(f"Experiment folder not found: {exp_root}")
